# This file shows the used workflow in the master thesis
## This a part of a master thesis by Mohamed Shaker

The file is also accessible on [Github](https://github.com/sherifix/my_thesis/blob/master/my_workflow.ipynb) for reproducibility.


**Important notes** 
- This is Python 3 based Kernel created on Jupyter Notebook
- Cell used bash command start with '%%bash'
- Single bash commands start with '!'
-The provided code runs after configuration of Jupyter server and environments creation steps shown on [README.md](https://github.com/sherifix/cel_screening/blob/master/README.md)  

In [4]:
# required Imports
import pandas as pd 


In [ ]:
"""
After downloading ThermoBase from https://togodb.org/db/thermobase
It was saved in data/thermobase_data
"""

In [5]:
# read thermobase 
df = pd.read_csv("data/thermobase_data/ThermoBase.csv")
organism = df['Name'].tolist()

# Create names.txt file with all organisms' name in thermobase
with open("data/thermobase_data/names.txt", 'w') as f:
    for name in organism:
        f.write(f"{name}\n")

#Count organism in thermobase
count = 0
with open("data/thermobase_data/names.txt", 'r') as f:
    for line in f.readlines():
        count+=1
    print(f" ==== Extracted {count} organism names from ThermoBase.csv ====")  

 ==== Extracted 1238 organism names from ThermoBase.csv ====


In [6]:
%%bash

# Use taxonkit tool to convert these name to TaxIDs compatible with NCBI 
# Use -f parametter for wider coverage

cd data/thermobase_data/
taxonkit name2taxid -f names.txt -o taxonkit_out.txt

09:56:18.034 [WARN] multiple TaxIds found for 'Ureibacillus composti'


In [14]:
%%bash 

# Check output from taxonkit 

cd data/thermobase_data/
head -5 taxonkit_out.txt
    
echo "=========================="
echo "Number of entries is"
wc -l taxonkit_out.txt

Methanopyrus kandleri 116	190192
Geogemma barossii 121	1927912
Pyrolobus fumarii 1A, DSM 11204	694429
Pyrococcus kukulkanii NCB100	1609559
Pyrodictium brockii S1	35616
Number of entries is
1239 taxonkit_out.txt


In [15]:
# TaxIDs were extracted to taxids.txt to enable batch download of the proteomes

df_taxonout = pd.read_csv("data/thermobase_data/taxonkit_out.txt",
                          sep='\t',
                          header=None)

valid_taxids = pd.to_numeric(df_taxonout.iloc[:,1], errors='coerce').dropna().astype(int).tolist()

with open("data/thermobase_data/taxids.txt", 'w') as f:
    for taxid in valid_taxids:
        f.write(f"{taxid}\n")

In [ ]:
%%bash

# Downloading reference sequence from GEnBank using datasets command
# Downloading script is in thesis_script/thermobase_proteome_download.sh

bash thesis_scripts/thermobase_proteome_download.sh

In [ ]:
%%bash 

# Unzipping all proteomes data while renaming each protein.faa fasta 
# to match the name of taxid_proteome 

for zip in *.zip; do
    # Get the base name without .zip
    basename="${zip%.zip}"
    
    # Create directory with that name
    mkdir -p "$basename"
    
    # Extract the .faa file into that directory and rename it
    unzip -j "$zip" "*/data/*.faa" -d "$basename"
    
    # Rename the extracted protein.faa to match directory name
    mv "$basename/protein.faa" "$basename/${basename}.faa" 2>/dev/null || true
    
    echo "Created: $basename/${basename}.faa"
done

In [32]:
%%bash


# Example output 
echo "Example architecture of proteomes directory"
echo  "File naming follow {taxid}_{proteome}.faa"
tree data/proteomes/1008305_GCA_041505935.1/


echo "=============================="
echo "=============================="
echo "Number of downloaded proteomes"
ls data/proteomes/ | wc -l 

Example architecture of proteomes directory
File naming follow {taxid}_{proteome}.faa
data/proteomes/1008305_GCA_041505935.1/
└── 1008305_GCA_041505935.1.faa

1 directory, 1 file
Number of downloaded proteomes
796


**=======================================================**

### Collecting needed GH families

Next step is to provide the needed GH families for building the **HMMER profiles**


After Reviewing multiple literature resources, GH families 6,7,9,12,44,45,48,5 are to be used. However, due to large number of entries in GH5, it was used based on the subfamily classifcation 

In [33]:
# Based on the developed pipeline, it is important to have one family per line
# And to add the families to GH_families.txt in data/ directory

families = """GH5_5
GH5_52
GH5_2
GH5_4
GH5_9
GH5_25
GH5_1
GH45
GH44
GH48
GH7
GH6
GH9
GH12"""

with open("data/GH_families.txt", "w") as f:
    f.write(families)

### Download dbCAN.hmm databases

Next step is to download dbCAN.hmm to extract catalytic domains for profile construction

In [ ]:
%%bash

# The following command create a directory for the database 
# then download the database and press it 

mkdir data/dbcan 
wget -O dbcan/dbCAN.hmm https://pro.unl.edu/dbCAN2/download/run_dbCAN_database_
total/dbCAN.hmm
hmmpress dbCAN.hmm

### Collecting reference structures 
A reference structure should be available for each family profile constructed to be used in structureal alignment step.

This step also is based on literature and databases review. 

The reference structures ideally would be collected from Protein Data Bank database to ensure high resolution. 

The files should be collected in data/reference_structures

After deciding what reference structures to be used, it was updated in the bash script responsible for downloading the reference structures

In [53]:
%%bash 

# This can be modifired in thesis_scripts/download_reference_structures.sh

grep -A 13 "declare -A STRUCTURES" thesis_scripts/download_reference_structures.sh

declare -A STRUCTURES=(
    ["gh12"]="3VGI"
    ["gh44"]="2E4T"
    ["gh45"]="5XBU"
    ["gh48"]="5YJ6"
    ["gh5_1"]="2ZUM"
    ["gh5_2"]="2CKS"
    ["gh5_25"]="7VT8"
    ["gh5_4"]="6MQ4"
    ["gh5_5"]="3QR3"
    ["gh6"]="4B4H"
    ["gh7"]="1CEL"
    ["gh9"]="1UT9"
)


In [ ]:
%%bash 

# Now run the script to get the needed reference_structures 

bash thesis_scripts/download_reference_structures.sh

In [55]:
%%bash

# The used reference structures
echo "Architecture of reference structures after running the script"
echo "========================================"
tree data/reference_structures/

Architecture of reference structures after running the script
data/reference_structures/
├── gh12
│   └── 3VGI.pdb
├── gh44
│   └── 2E4T.pdb
├── gh45
│   └── 5xBU.pdb
├── gh48
│   └── 5YJ6.pdb
├── gh5_1
│   └── 2ZUM.pdb
├── gh5_2
│   └── 2CKS.pdb
├── gh5_25
│   └── 7VT8.pdb
├── gh5_4
│   └── 6MQ4.pdb
├── gh5_5
│   └── 3QR3.pdb
├── gh6
│   └── 4B4H.pdb
├── gh7
│   └── 1CEL.pdb
└── gh9
    └── 1UT9.pdb

13 directories, 12 files


### Universal ligand for molecular docking

After literature review of multiple resources, cellotetraose was used as universal docking to evaluate different cellulase candidates binding affinity

Cellotetraose was downloaded from PubChem using wget

It was placed in the required directory to run the pipeline

In [57]:
%%bash 

mkdir results/autodock_vina/
    
wget -O results/autodock_vina/cellotetraose.sdf "https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/0002988D00000001/SDF?response_type=display"

mkdir: cannot create directory ‘results/autodock_vina/’: File exists
--2026-06-11 11:12:08--  https://pubchem.ncbi.nlm.nih.gov/rest/pug/conformers/0002988D00000001/SDF?response_type=display
Resolving pubchem.ncbi.nlm.nih.gov (pubchem.ncbi.nlm.nih.gov)... 130.14.29.110, 2607:f220:41e:4290::110
Connecting to pubchem.ncbi.nlm.nih.gov (pubchem.ncbi.nlm.nih.gov)|130.14.29.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘results/autodock_vina/cellotetraose.sdf’

     0K ..........                                             98.7M=0s

2026-06-11 11:12:09 (98.7 MB/s) - ‘results/autodock_vina/cellotetraose.sdf’ saved [10619]



### Run pipeline

All the required files are downloaded and located in the locations required by the pipeline as stated in README.me 

To run the pipeline, execute the following command

In [59]:
%%bash 

snakemake --cores 8 --dry-run

Keeping GH9: 136 sequences
Keeping GH45: 74 sequences
Keeping GH44: 11 sequences
Keeping GH5_5: 53 sequences
Keeping GH6: 65 sequences
Keeping GH12: 67 sequences
Keeping GH5_2: 103 sequences
Keeping GH7: 76 sequences
Keeping GH48: 23 sequences
Keeping GH5_1: 35 sequences
Keeping GH5_25: 11 sequences
Keeping GH5_4: 47 sequences
Building DAG of jobs...
Job stats:
job                                count
-------------------------------  -------
all                                    1
calculate_pi                           1
filter_overlapping_domains             1
filter_top_hits                        1
filtering_thermo_signal_results        1
hmmsearch_parsing                      1
parse_p2rank_results                   1
parse_usalign                          1
parse_vina_results                     1
prepare_docked_fasta                   1
prepare_ligand                         1
prepare_pdb_lists                      1
prepare_receptors                      1
prepare_structures   

**-----------------------------------------------------**
### Analysing pipeline's output files

After running the pipeline and getting all the output files 

All Visualizations files from US-align, P2Rank, AutoDock Vina were exported to windows machine and visualized using PyMOL open source 

After inspecting the visulaization files from the docking experiment by Autodock Vina, all hydrogen bond residues were collected and merged to one file <mark>output_files/vina_polar_contacts.csv</mark>

### Creating figures and tables 

Now all the data and output results are available.

Figures and tables were created. The source code of figures and tables is shown in [figures_tables_generator.ipynb](https://github.com/sherifix/my_thesis/blob/master/figures_tables_generator.ipynb)
